# **3. Feature Set Definition**

## **3.1 Import libraries and define paths**

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
processed_dir = Path("../data/processed")
dataset_path = processed_dir / "era5_supervised_preprocessed.csv"
full_feature_list_path = processed_dir / "feature_columns.txt"

features_baseline_path = processed_dir / "features_baseline.txt"
features_selected_history_path = processed_dir / "features_selected_history.txt"
features_full_path = processed_dir / "features_full.txt"

## **3.2 Load available features**

In [3]:
feature_columns = full_feature_list_path.read_text(encoding="utf-8").splitlines()
dataset_columns = pd.read_csv(dataset_path, nrows=0).columns.tolist()

print("Available model features:", len(feature_columns))
print("Total dataset columns:", len(dataset_columns))

Available model features: 92
Total dataset columns: 100


## **3.3 Feature set A: baseline**

Uses current hour meteorological conditions and time variables only

In [4]:
features_baseline = [
    "t2m_c",
    "d2m_c",
    "msl_hpa",
    "sp_hpa",
    "tcc",
    "u10",
    "v10",
    "wind_speed_10m",
    "month",
    "dayofyear",
    "hour",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
]

len(features_baseline)

15

## **3.4 Feature set B: selected history**

Adds a smaller number of lagged and rolling variables.

In [5]:
selected_history_features = [
    "msl_hpa_lag1",
    "msl_hpa_lag3",
    "msl_hpa_lag6",
    "d2m_c_lag1",
    "d2m_c_lag3",
    "tcc_lag1",
    "tcc_lag3",
    "wind_speed_10m_lag1",
    "wind_speed_10m_lag3",
    "tp_mm_lag1",
    "tp_mm_lag3",
    "tp_mm_lag6",
    "msl_hpa_roll6_mean",
    "d2m_c_roll6_mean",
    "tcc_roll6_mean",
    "wind_speed_10m_roll6_mean",
    "wind_speed_10m_roll6_max",
    "tp_mm_roll6_sum",
    "tp_mm_roll24_sum",
]

features_selected_history = features_baseline + selected_history_features
len(features_selected_history)

34

## **3.5 Feature set C: full**

all engineered model features from notebook 02

In [6]:
features_full = feature_columns.copy()
len(features_full)

92

## **3.6 Validate feature sets**
target leakage is a common reason in ML papers which gets invalidated. 

In [8]:
feature_sets = {
    "baseline": features_baseline,
    "selected_history": features_selected_history,
    "full": features_full,
}

targets = {"extreme_precip_975", "extreme_precip_99"}
metadata = {"valid_time", "latitude", "longitude", "tp_mm", "season", "split"}

for name, features in feature_sets.items():
    feature_set = set(features)

    missing = feature_set - set(dataset_columns)
    duplicates = {f for f in features if features.count(f) > 1}
    target_leakage = feature_set & targets
    metadata_leakage = feature_set & metadata

    assert not target_leakage, f"{name}: target leakage detected ({target_leakage})"
    assert not metadata_leakage, (
        f"{name}: metadata columns in features ({metadata_leakage})"
    )
    assert not missing, f"{name}: columns not in dataset ({missing})"
    assert not duplicates, f"{name}: duplicate features ({duplicates})"

    print(f"  {name}: {len(features)} features")

print("\nAll featuresets here validated successfully")

  baseline: 15 features
  selected_history: 34 features
  full: 92 features

All featuresets here validated successfully


## **3.8 Save feature lists**

In [9]:
feature_set_paths = {
    "baseline": features_baseline_path,
    "selected_history": features_selected_history_path,
    "full": features_full_path,
}

for name, path in feature_set_paths.items():
    features = feature_sets[name]
    path.write_text("\n".join(features), encoding="utf-8")
    print(f"Saved {name}: {len(features)} features in {path}")

Saved baseline: 15 features in ..\data\processed\features_baseline.txt
Saved selected_history: 34 features in ..\data\processed\features_selected_history.txt
Saved full: 92 features in ..\data\processed\features_full.txt
